# Synthetic Data Generator
### Step 9 : Row Rebuilder

## Overview

This notebook supports the synthetic pipeline stage documented by this notebook. It is part of the Synthetic support portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook supports the synthetic pipeline stage documented by this notebook.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Clarify whether the notebook is a support, testing, streaming, or Bronze-handoff step in the synthetic path.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/synthetic_09_row_rebuilder_code_reference.md`


In [1]:
import os

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.row_rebuilder import (
    rebuild_consumed_messages_to_observations,
)

from utils.core.env_helpers import (
    env_bool, 
    env_int, 
    env_str,
)

In [2]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

NUMBER_OF_SENSORS = env_int("SYNTHETIC_SENSOR_COUNT", 52)

OBSERVATION_BATCH_SIZE = env_int(
    "SYNTHETIC_OBSERVATION_BATCH_SIZE",
    500,
    aliases=("OBSERVATION_BATCH_SIZE",),
)

REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME = env_str(
    "SYNTHETIC_CONSUMED_MESSAGES_TABLE",
    "synthetic_sensor_messages_consumed_stage",
    aliases=("CONSUMER_TARGET_TABLE",),
)

REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME = env_str(
    "SYNTHETIC_REBUILT_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_rebuilt_stage",
)

REBUILD_STATUS = env_str("SYNTHETIC_REBUILD_STATUS", "pending")

COMPLETE_ONLY_FLAG = env_bool("SYNTHETIC_REBUILD_COMPLETE_ONLY", True)
MARK_SOURCE_REBUILT_FLAG = env_bool("SYNTHETIC_MARK_SOURCE_REBUILT", True)

OBSERVATION_WINDOW_SIZE = env_int(
    "SYNTHETIC_REBUILD_OBSERVATION_WINDOW_SIZE",
    2500,
)

print("Row rebuild config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("source table:", REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME)
print("target table:", REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME)
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("rebuild observation window:", OBSERVATION_WINDOW_SIZE)
print("expected sensors per complete observation:", NUMBER_OF_SENSORS)

Row rebuild config
schema: capstone
dataset id: pump_synthetic_v1
run id: synthetic_run_001
source table: synthetic_sensor_messages_consumed_stage
target table: synthetic_sensor_observations_rebuilt_stage
observation batch size: 500
rebuild observation window: 2500
expected sensors per complete observation: 52


---

In [3]:

engine = get_engine_from_env()


---

In [4]:


rebuild_result = rebuild_consumed_messages_to_observations(
    engine=engine,
    schema=SCHEMA,
    source_table=REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME,
    target_table=REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    rebuild_status=REBUILD_STATUS,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    mark_source_rebuilt=MARK_SOURCE_REBUILT_FLAG,
    
    observation_window_size=OBSERVATION_WINDOW_SIZE,
)

display(rebuild_result)

[obs-window] 1 | observation_index 1 to 2,500
[rebuild] Added 52 new columns to capstone.synthetic_sensor_observations_rebuilt_stage
[obs-window] 2 | observation_index 2,501 to 5,000
[obs-window] 3 | observation_index 5,001 to 7,500
[obs-window] 4 | observation_index 7,501 to 10,000
[obs-window] 5 | observation_index 10,001 to 12,500
[obs-window] 6 | observation_index 12,501 to 15,000
[obs-window] 7 | observation_index 15,001 to 17,500
[obs-window] 8 | observation_index 17,501 to 20,000
[obs-window] 9 | observation_index 20,001 to 22,500
[obs-window] 10 | observation_index 22,501 to 25,000
[obs-window] 11 | observation_index 25,001 to 27,500
[obs-window] 12 | observation_index 27,501 to 30,000
[obs-window] 13 | observation_index 30,001 to 32,500
[obs-window] 14 | observation_index 32,501 to 35,000
[obs-window] 15 | observation_index 35,001 to 37,500
[obs-window] 16 | observation_index 37,501 to 40,000
[obs-window] 17 | observation_index 40,001 to 42,500
[obs-window] 18 | observation_in

{'status': 'rebuilt',
 'consumed_rows': 11700000,
 'deduped_rows': 11700000,
 'rebuilt_rows': 225000,
 'rebuilt_observations': 225000,
 'updated_source_observation_groups': 225000,
 'target_table': 'synthetic_sensor_observations_rebuilt_stage'}

----

In [5]:
validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS rebuilt_row_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS complete_row_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        MIN(observation_timestamp) AS min_observation_timestamp,
        MAX(observation_timestamp) AS max_observation_timestamp
    FROM {SCHEMA}.{REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME}
    """
)

display(validation_dataframe)

,rebuilt_row_count,complete_row_count,min_observation_index,max_observation_index,min_observation_timestamp,max_observation_timestamp
0,225000,225000,1,225000,2026-04-16 00:00:00+00:00,2026-09-19 05:59:00+00:00


---

In [6]:
sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        observation_timestamp,
        stream_state,
        phase,
        meta_episode_id,
        meta_primary_fault_type,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        rebuild_sensor_count,
        rebuild_is_complete
    FROM {SCHEMA}.{REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME}
    ORDER BY observation_index
    LIMIT 10
    """
)
display(sample_dataframe)

,dataset_id,run_id,asset_id,observation_index,observation_timestamp,stream_state,phase,meta_episode_id,meta_primary_fault_type,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,rebuild_sensor_count,rebuild_is_complete
0,pump_synthetic_v1,synthetic_run_001,pump_asset_001,1,2026-04-16 00:00:00+00:00,normal,normal,0,drift_down,2.344329,48.327756,53.575950,43.424633,617.854335,52,True
1,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2,2026-04-16 00:01:00+00:00,normal,normal,0,drift_down,2.316564,49.037209,50.378763,47.447264,617.673971,52,True
2,pump_synthetic_v1,synthetic_run_001,pump_asset_001,3,2026-04-16 00:02:00+00:00,normal,normal,0,drift_down,2.449135,48.920071,52.446083,46.391672,617.791244,52,True
3,pump_synthetic_v1,synthetic_run_001,pump_asset_001,4,2026-04-16 00:03:00+00:00,normal,normal,0,drift_down,2.489110,49.304505,51.872589,42.288103,617.794481,52,True
4,pump_synthetic_v1,synthetic_run_001,pump_asset_001,5,2026-04-16 00:04:00+00:00,normal,normal,0,drift_down,2.364396,45.769864,51.927495,41.465983,617.993983,52,True
5,pump_synthetic_v1,synthetic_run_001,pump_asset_001,6,2026-04-16 00:05:00+00:00,normal,normal,0,drift_down,2.223095,47.166849,50.717235,40.841192,617.698302,52,True
6,pump_synthetic_v1,synthetic_run_001,pump_asset_001,7,2026-04-16 00:06:00+00:00,normal,normal,0,drift_down,2.466576,48.560256,52.010692,44.414286,618.282081,52,True
7,pump_synthetic_v1,synthetic_run_001,pump_asset_001,8,2026-04-16 00:07:00+00:00,normal,normal,0,drift_down,2.337132,43.087177,52.973015,44.378573,618.379612,52,True
8,pump_synthetic_v1,synthetic_run_001,pump_asset_001,9,2026-04-16 00:08:00+00:00,normal,normal,0,drift_down,2.355965,45.514778,55.131609,43.304947,618.339956,52,True
9,pump_synthetic_v1,synthetic_run_001,pump_asset_001,10,2026-04-16 00:09:00+00:00,normal,normal,0,drift_down,2.270712,46.130963,53.648135,42.564812,617.956074,52,True


----

In [7]:
incomplete_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        rebuild_sensor_count,
        rebuild_is_complete,
        rebuild_notes
    FROM "{SCHEMA}"."{REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME}"
    WHERE rebuild_is_complete = FALSE
    ORDER BY observation_index
    LIMIT 25
    """,
)

display(incomplete_dataframe)

,dataset_id,run_id,asset_id,observation_index,rebuild_sensor_count,rebuild_is_complete,rebuild_notes


----

In [8]:
stage_9_progress_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS rebuilt_observation_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS complete_rebuild_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = FALSE) AS incomplete_rebuild_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        COUNT(DISTINCT observation_index) AS distinct_observation_count
    FROM "{SCHEMA}"."synthetic_sensor_observations_rebuilt_stage"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_9_progress_dataframe)

,rebuilt_observation_count,complete_rebuild_count,incomplete_rebuild_count,min_observation_index,max_observation_index,distinct_observation_count
0,225000,225000,0,1,225000,225000


In [9]:
consumed_rebuild_status_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        rebuild_status,
        COUNT(*) AS row_count
    FROM "{SCHEMA}"."synthetic_sensor_messages_consumed_stage"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY rebuild_status
    ORDER BY row_count DESC
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(consumed_rebuild_status_dataframe)

,rebuild_status,row_count
0,rebuilt,11700000


In [10]:
stage_9_final_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS rebuilt_observation_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS complete_rebuild_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = FALSE) AS incomplete_rebuild_count,
        COUNT(*) FILTER (WHERE rebuild_sensor_count = 52) AS full_sensor_count_rows,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        (
            COUNT(*) = 225000
            AND COUNT(DISTINCT observation_index) = 225000
            AND COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) = 225000
            AND COUNT(*) FILTER (WHERE rebuild_sensor_count = 52) = 225000
            AND MIN(observation_index) = 1
            AND MAX(observation_index) = 225000
        ) AS ready_for_stage_10
    FROM "{SCHEMA}"."synthetic_sensor_observations_rebuilt_stage"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_9_final_validation_dataframe)

,rebuilt_observation_count,distinct_observation_count,complete_rebuild_count,incomplete_rebuild_count,full_sensor_count_rows,min_observation_index,max_observation_index,ready_for_stage_10
0,225000,225000,225000,0,225000,1,225000,True


In [11]:
# Dispose Engine
engine.dispose()

## Summary

This notebook preserves the current analytical behavior while documenting the role of the Synthetic support step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

The row rebuilder reconstructs wide observation rows from consumed sensor messages.
